In [ ]:
%pip install -U mlflow langchain langchain-community langchain-text-splitters pypdf
dbutils.library.restartPython()

In [0]:
import os
import json
import glob
import pickle
import mlflow
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

CONFIG_PATH = "/Volumes/main/default/dino_pdfs/project/config/project_config.json"
with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

VOLUME_URI = config["volume_uri"]
ARTIFACTS_URI = config["artifacts_uri"]
TABLE_NAME = config["delta_table"]
CHUNK_SIZE = int(config["chunk_size"])
CHUNK_OVERLAP = int(config["chunk_overlap"])

pdf_files = sorted(glob.glob(VOLUME_URI + "/*.pdf"))
mlflow.set_experiment(config["mlflow_experiment"])

print("PDF count:", len(pdf_files))
print("Delta table:", TABLE_NAME)

PDF count: 8
Delta table: main.default.dinosaur_chunks


In [0]:
from langchain_community.document_loaders import PyPDFLoader

all_docs = []

for f in dbutils.fs.ls(VOLUME_URI):
    if not f.isDir() and f.name.lower().endswith(".pdf"):
        # Convert dbfs:/ URI to /Volumes/ path for PyPDFLoader
        volume_path = f.path.replace('dbfs:', '')
        
        loader = PyPDFLoader(volume_path)
        docs = loader.load()

        for d in docs:
            d.metadata["source_file"] = f.name
            d.metadata["source_path"] = f.path
            d.metadata["domain"] = "dinosaurs"

        all_docs.extend(docs)

print("Total pages loaded:", len(all_docs))

Total pages loaded: 47


In [0]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunked_docs = splitter.split_documents(all_docs)

for i, d in enumerate(chunked_docs):
    d.metadata["chunk_id"] = i

print("Total chunks created:", len(chunked_docs))
if chunked_docs:
    print("Example chunk metadata:", chunked_docs[0].metadata)
    print("Example chunk preview:", chunked_docs[0].page_content[:400])

Total chunks created: 167
Example chunk metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-27T18:15:30-04:00', 'author': 'rohit sharma', 'moddate': '2026-03-27T18:15:30-04:00', 'source': '/Volumes/main/default/dino_pdfs/Document1_Dinosaur_Classification_and_General_Characteristics.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'Document1_Dinosaur_Classification_and_General_Characteristics.pdf', 'source_path': 'dbfs:/Volumes/main/default/dino_pdfs/Document1_Dinosaur_Classification_and_General_Characteristics.pdf', 'domain': 'dinosaurs', 'chunk_id': 0}
Example chunk preview: 1 
Dinosaur Classification and General Characteristics 
Dinosaur classification refers to the scientific organization of dinosaurs based on 
their evolutionary relationships and anatomical features of dinosaurs. These articles 
serve as one of the most widely consulted, collaboratively edited resources 
summarizin

In [0]:
chunks_pickle_path = os.path.join(ARTIFACTS_URI, "chunked_docs.pkl")
with open(chunks_pickle_path, "wb") as f:
    pickle.dump(chunked_docs, f)

print("Saved chunk objects to:", chunks_pickle_path)

Saved chunk objects to: /Volumes/main/default/dino_pdfs/project/artifacts/chunked_docs.pkl


In [0]:
rows = []
for d in chunked_docs:
    rows.append({
        "chunk_id": int(d.metadata.get("chunk_id", -1)),
        "source_file": str(d.metadata.get("source_file", "")),
        "page": int(d.metadata.get("page", -1)) if d.metadata.get("page") is not None else -1,
        "domain": str(d.metadata.get("domain", "dinosaurs")),
        "source_path": str(d.metadata.get("source_path", "")),
        "text": d.page_content
    })

chunks_df = spark.createDataFrame(rows)
chunks_df.write.mode("overwrite").format("delta").saveAsTable(TABLE_NAME)

display(spark.sql(f'''
SELECT source_file, COUNT(*) AS chunk_count
FROM {TABLE_NAME}
GROUP BY source_file
ORDER BY chunk_count DESC
'''))

source_file,chunk_count
Document1_Dinosaur_Classification_and_General_Characteristics.pdf,38
Document5_Dinosaur_Physiology_Behavior_and_Extinction_Events.pdf,23
Document4_Dinosaur_Communication_Reproduction_and_Physiology.pdf,22
Document6_Dinosaur_Reproduction_Parental_Care_and_Migration Behavior.pdf,20
Document3_Dinosaur_Classification_Size_and_Behavior.pdf,19
Document2_History_of_Dinosaur_Discovery.pdf,19
Document7_Dinosaur_Sounds_Communication_and_Social_Interaction.pdf,14
Document8_Common_Misconceptions_About_Dinosaurs.pdf,12


In [0]:
with mlflow.start_run(run_name="01_data_ingestion_and_chunking"):
    mlflow.log_param("chunk_size", CHUNK_SIZE)
    mlflow.log_param("chunk_overlap", CHUNK_OVERLAP)
    mlflow.log_param("delta_table", TABLE_NAME)
    mlflow.log_metric("num_pdf_files", len(pdf_files))
    mlflow.log_metric("num_pages", len(all_docs))
    mlflow.log_metric("num_chunks", len(chunked_docs))

summary_df = pd.DataFrame([{
    "num_pdf_files": len(pdf_files),
    "num_pages": len(all_docs),
    "num_chunks": len(chunked_docs),
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP
}])

display(summary_df)
print("Chunking run logged to MLflow.")

num_pdf_files,num_pages,num_chunks,chunk_size,chunk_overlap
8,47,167,800,150


Chunking run logged to MLflow.
